# Evaluasi Model Random Forest

Notebook ini menjalankan audit dataset, preprocessing, multilabel cross-validation, evaluasi out-of-fold, visualisasi hasil, permutation importance, dan penyimpanan artifact model.

## Tujuan pengujian

Model yang diuji: **Random Forest**.

Target `Perlu` dan `Tidak Perlu` berasal dari Dataset_Model. Kolom post-test tidak digunakan sebagai feature untuk mencegah data leakage.

> Model ini menguji hubungan non-linear dan interaksi antar-feature pada data tabular.

In [ ]:
from pathlib import Path
import json
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    confusion_matrix, f1_score, hamming_loss,
    precision_score, recall_score, roc_auc_score,
    make_scorer,
)
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
except ImportError as exc:
    raise ImportError(
        "Install dependency first with: "
        "%pip install iterative-stratification"
    ) from exc

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

print("Python:", sys.executable)
print("Working directory:", Path.cwd())


In [ ]:
DATA_PATH = Path("Dataset_Heroism.xlsx")
if not DATA_PATH.exists() and Path.cwd().name == "notebooks":
    DATA_PATH = Path("..").resolve() / "Dataset_Heroism.xlsx"

SHEET = "Dataset_Model"
ARTIFACT_DIR = Path("artifacts")
REPORT_DIR = Path("reports")
ARTIFACT_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

FEATURES_NUMERIC = [
    "Usia", "IF_Pre", "OMD_Pre", "Potensi_Pre",
    "Eksplorasi_Pre", "Refleksi_Pre", "Perencanaan_Pre",
]
FEATURES_CATEGORICAL = ["Jenis_Kelamin", "Terlibat_Memulung"]
FEATURES = FEATURES_NUMERIC + FEATURES_CATEGORICAL
TARGETS = [
    "Y_Potensi", "Y_Eksplorasi",
    "Y_Refleksi", "Y_Perencanaan",
]

LABEL_TO_INT = {"Tidak Perlu": 0, "Perlu": 1}

df = pd.read_excel(DATA_PATH, sheet_name=SHEET)
X = df[FEATURES].copy()
Y = df[TARGETS].replace(LABEL_TO_INT).astype(int)

print("Dataset:", DATA_PATH)
print("Shape:", df.shape)
display(df.head())
display(df[TARGETS].apply(lambda s: s.value_counts()))


In [ ]:
required = FEATURES + TARGETS
missing_columns = [c for c in required if c not in df.columns]
assert not missing_columns, f"Kolom hilang: {missing_columns}"
assert df[required].isna().sum().sum() == 0, "Ada missing value."
assert not df.duplicated().any(), "Ada duplikasi baris."
assert set(df["Jenis_Kelamin"]).issubset({"P", "L"})
assert set(df["Terlibat_Memulung"]).issubset({"Ya", "Tidak"})
for target in TARGETS:
    assert set(df[target]).issubset(set(LABEL_TO_INT)), f"Target invalid: {target}"

range_rules = {
    "Usia": (8, 13),
    "IF_Pre": (15, 60),
    "OMD_Pre": (15, 60),
    "Potensi_Pre": (10, 40),
    "Eksplorasi_Pre": (10, 40),
    "Refleksi_Pre": (10, 40),
    "Perencanaan_Pre": (10, 40),
}
for col, (lo, hi) in range_rules.items():
    assert df[col].between(lo, hi).all(), f"Rentang invalid: {col}"

print("Audit data: PASS")
print("Missing cells:", int(df[required].isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("Target distribution:")
display(Y.apply(pd.Series.value_counts).fillna(0).astype(int))


In [ ]:
preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]),
        FEATURES_NUMERIC,
    ),
    (
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]),
        FEATURES_CATEGORICAL,
    ),
])

def positive_probabilities(model, X_input):
    probabilities = model.predict_proba(X_input)
    return np.column_stack([
        p[:, 1] if p.shape[1] > 1 else np.zeros(len(X_input))
        for p in probabilities
    ])

def evaluate_predictions(y_true, y_pred, y_prob=None):
    result = {
        "subset_accuracy": accuracy_score(y_true, y_pred),
        "hamming_loss": hamming_loss(y_true, y_pred),
        "macro_f1": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "macro_precision": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "macro_recall": recall_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
    }
    for i, target in enumerate(TARGETS):
        result[f"{target}_f1"] = f1_score(
            y_true[:, i], y_pred[:, i], zero_division=0
        )
        result[f"{target}_precision"] = precision_score(
            y_true[:, i], y_pred[:, i], zero_division=0
        )
        result[f"{target}_recall"] = recall_score(
            y_true[:, i], y_pred[:, i], zero_division=0
        )
        result[f"{target}_balanced_accuracy"] = balanced_accuracy_score(
            y_true[:, i], y_pred[:, i]
        )
        if y_prob is not None:
            try:
                result[f"{target}_roc_auc"] = roc_auc_score(
                    y_true[:, i], y_prob[:, i]
                )
            except ValueError:
                result[f"{target}_roc_auc"] = np.nan
    return result


In [ ]:
MODEL_NAME = "Random Forest"
estimator = MultiOutputClassifier(
    __import__("sklearn.ensemble", fromlist=["RandomForestClassifier"]).RandomForestClassifier(
        n_estimators=300,
        max_features="sqrt",
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )
)
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", estimator),
])


In [ ]:
cv_rows = []
oof_probability_sum = np.zeros((len(X), len(TARGETS)))
oof_count = np.zeros(len(X))

for seed in range(5):
    cv = MultilabelStratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=seed,
    )

    for fold, (train_idx, test_idx) in enumerate(
        cv.split(X, Y), start=1
    ):
        fold_model = clone(pipeline)
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train = Y.iloc[train_idx].to_numpy()
        y_test = Y.iloc[test_idx].to_numpy()

        fold_model.fit(X_train, y_train)
        y_pred = fold_model.predict(X_test)
        y_prob = positive_probabilities(fold_model, X_test)

        oof_probability_sum[test_idx] += y_prob
        oof_count[test_idx] += 1

        metrics = evaluate_predictions(y_test, y_pred, y_prob)
        metrics.update({
            "seed": seed,
            "fold": fold,
            "model": MODEL_NAME,
        })
        cv_rows.append(metrics)

cv_results = pd.DataFrame(cv_rows)
oof_prob = oof_probability_sum / oof_count[:, None]
oof_pred = (oof_prob >= 0.50).astype(int)

oof_metrics = evaluate_predictions(
    Y.to_numpy(), oof_pred, oof_prob
)

print("Cross-validation selesai.")
display(
    cv_results[
        ["seed", "fold", "macro_f1", "macro_recall",
         "hamming_loss", "subset_accuracy"]
    ].head()
)
display(pd.DataFrame([oof_metrics], index=[MODEL_NAME]))


## Ringkasan evaluasi

Tinjau macro-F1, macro-recall, Hamming loss, subset accuracy, dan variasi antar-fold sebelum menyimpulkan model. Jangan menggunakan akurasi saja.

In [ ]:
display(cv_results.groupby("model")[
    ["macro_f1", "macro_precision", "macro_recall", "hamming_loss", "subset_accuracy"]
].agg(["mean", "std"]).T)


In [ ]:
# 1. Distribusi label target
target_counts = (
    df[TARGETS]
    .replace({0: "Tidak Perlu", 1: "Perlu"})
    .melt(var_name="Target", value_name="Label")
)

plt.figure(figsize=(11, 5))
sns.countplot(
    data=target_counts,
    x="Target",
    hue="Label",
    palette="Set2",
)
plt.title(f"Distribusi Target - {MODEL_NAME}")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

# 2. Metrik cross-validation
metric_names = ["macro_f1", "macro_precision", "macro_recall"]
metric_means = cv_results[metric_names].mean().sort_values(ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(
    x=metric_means.index,
    y=metric_means.values,
    color="#2F6B9A",
)
plt.ylim(0, 1)
plt.title(f"Rata-rata Metrik Cross-Validation - {MODEL_NAME}")
plt.ylabel("Score")
plt.xlabel("")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

# 3. Confusion matrix out-of-fold per target
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, i, target in zip(axes.ravel(), range(4), TARGETS):
    cm = confusion_matrix(
        Y.iloc[:, i],
        oof_pred[:, i],
        labels=[0, 1],
    )
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        ax=ax,
        xticklabels=["Tidak Perlu", "Perlu"],
        yticklabels=["Tidak Perlu", "Perlu"],
    )
    ax.set_title(target)
    ax.set_xlabel("Prediksi")
    ax.set_ylabel("Aktual")
plt.suptitle(f"Confusion Matrix Out-of-Fold - {MODEL_NAME}", y=1.02)
plt.tight_layout()
plt.show()

# 4. Probabilitas kebutuhan penguatan
probability_df = pd.DataFrame(
    oof_prob,
    columns=TARGETS,
)
plt.figure(figsize=(11, 5))
for target in TARGETS:
    sns.kdeplot(
        probability_df[target],
        label=target,
        fill=False,
    )
plt.axvline(0.50, color="black", linestyle="--", label="Threshold 0.50")
plt.xlim(0, 1)
plt.title(f"Distribusi Probabilitas Out-of-Fold - {MODEL_NAME}")
plt.xlabel("Probabilitas Perlu")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Refit pada seluruh data untuk artifact dan interpretasi.
final_model = clone(pipeline)
final_model.fit(X, Y.to_numpy())

# Permutation importance pada feature asli.
scorer = make_scorer(
    f1_score,
    average="macro",
    zero_division=0,
)
permutation = __import__("sklearn.inspection", fromlist=["permutation_importance"]).permutation_importance(
    final_model,
    X,
    Y.to_numpy(),
    scoring=scorer,
    n_repeats=20,
    random_state=42,
)

importance = pd.DataFrame({
    "feature": FEATURES,
    "importance_mean": permutation.importances_mean,
    "importance_std": permutation.importances_std,
}).sort_values("importance_mean", ascending=False)

plt.figure(figsize=(9, 5))
sns.barplot(
    data=importance.head(12),
    y="feature",
    x="importance_mean",
    color="#2F6B9A",
)
plt.title(f"Permutation Importance - {MODEL_NAME}")
plt.xlabel("Mean decrease in macro-F1")
plt.ylabel("")
plt.tight_layout()
plt.show()

display(importance)


In [ ]:
evaluation_summary = pd.DataFrame([{
    "model": MODEL_NAME,
    "macro_f1_oof": oof_metrics["macro_f1"],
    "macro_precision_oof": oof_metrics["macro_precision"],
    "macro_recall_oof": oof_metrics["macro_recall"],
    "hamming_loss_oof": oof_metrics["hamming_loss"],
    "subset_accuracy_oof": oof_metrics["subset_accuracy"],
    "cv_macro_f1_mean": cv_results["macro_f1"].mean(),
    "cv_macro_f1_std": cv_results["macro_f1"].std(),
}])

safe_name = MODEL_NAME.lower().replace(" ", "_")
cv_results.to_csv(
    REPORT_DIR / f"{safe_name}_cv_results.csv",
    index=False,
)
evaluation_summary.to_csv(
    REPORT_DIR / f"{safe_name}_summary.csv",
    index=False,
)
importance.to_csv(
    REPORT_DIR / f"{safe_name}_feature_importance.csv",
    index=False,
)

joblib.dump(
    final_model,
    ARTIFACT_DIR / f"model_{safe_name}.pkl",
)

metadata = {
    "model": MODEL_NAME,
    "data_path": str(DATA_PATH),
    "sheet": SHEET,
    "features": FEATURES,
    "targets": TARGETS,
    "label_mapping": LABEL_TO_INT,
    "cv": {
        "type": "MultilabelStratifiedKFold",
        "folds": 5,
        "repeats": 5,
        "probability_threshold": 0.50,
    },
    "dataset_rows": int(len(df)),
    "deployment_status": "experimental_not_for_apk",
    "deployment_eligible": False,
    "target_definition": "Proxy: post-test score < 30; threshold requires validation.",
    "baseline_rule": "Same-aspect pre-test score < 30; compare on matched folds.",
    "warning": (
        "Do not deploy in Menyala-HERO: n=51, proxy threshold <30, "
        "open instrument crosswalk, and no established gain over rule baseline."
    ),
}

(REPORT_DIR / f"{safe_name}_metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False)
)

print("Artifact disimpan:")
print(ARTIFACT_DIR / f"model_{safe_name}.pkl")
print(REPORT_DIR / f"{safe_name}_summary.csv")


In [ ]:
from sklearn.metrics import fbeta_score

# Threshold tuning hanya untuk analisis keputusan.
# Dataset dan label asli tidak diubah.
threshold_grid = np.arange(0.30, 0.71, 0.05)
threshold_rows = []

for i, target in enumerate(TARGETS):
    y_true = Y.iloc[:, i].to_numpy()
    for threshold in threshold_grid:
        y_pred = (oof_prob[:, i] >= threshold).astype(int)
        threshold_rows.append({
            "target": target,
            "threshold": round(float(threshold), 2),
            "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
            "recall_perlu": recall_score(y_true, y_pred, zero_division=0),
            "precision_perlu": precision_score(y_true, y_pred, zero_division=0),
        })

threshold_results = pd.DataFrame(threshold_rows)
display(
    threshold_results.sort_values(
        ["target", "f2"], ascending=[True, False]
    ).groupby("target").head(3)
)

# Kandidat threshold terbaik berdasarkan F2.
best_thresholds = (
    threshold_results.sort_values(
        ["target", "f2"], ascending=[True, False]
    )
    .groupby("target", as_index=False)
    .first()
)
display(best_thresholds)

# Zona monitor adalah kebijakan review, bukan ground truth.
# Nilai ini dapat diubah setelah validasi dengan pendamping dan cohort baru.
MONITOR_LOW = 0.40
MONITOR_HIGH = 0.60

def triage(probability):
    if probability < MONITOR_LOW:
        return "Belum Menjadi Prioritas Penguatan"
    if probability <= MONITOR_HIGH:
        return "Perlu Monitor Lebih Lanjut"
    return "Perlu Penguatan Lanjutan"

triage_output = pd.DataFrame({
    target: [triage(p) for p in oof_prob[:, i]]
    for i, target in enumerate(TARGETS)
})

print("Distribusi triage pada out-of-fold probability:")
display(triage_output.apply(pd.Series.value_counts).fillna(0).astype(int))

plt.figure(figsize=(11, 5))
for target in TARGETS:
    sns.kdeplot(probability_df[target], label=target, fill=False)
plt.axvspan(
    MONITOR_LOW, MONITOR_HIGH,
    color="gold", alpha=0.18, label="Zona monitor",
)
plt.axvline(0.50, color="black", linestyle="--", label="Cutoff 0.50")
plt.xlim(0, 1)
plt.title(f"Zona Triage Probabilitas - {MODEL_NAME}")
plt.xlabel("Probabilitas Perlu Penguatan")
plt.legend()
plt.tight_layout()
plt.show()

triage_config = {
    "model": MODEL_NAME,
    "monitor_low": MONITOR_LOW,
    "monitor_high": MONITOR_HIGH,
    "threshold_candidates": best_thresholds.to_dict(orient="records"),
    "warning": (
        "Threshold kandidat berasal dari out-of-fold probability "
        "dan perlu validasi pada cohort baru."
    ),
}
(REPORT_DIR / f"{safe_name}_triage_thresholds.json").write_text(
    json.dumps(triage_config, indent=2, ensure_ascii=False)
)
print("Konfigurasi triage disimpan sebagai kandidat, bukan keputusan final.")


## Pembanding aturan sederhana (baseline operasional)

Model terlatih harus dibandingkan dengan aturan yang transparan: klasifikasikan aspek sebagai `Perlu` bila skor pre-test aspek yang sama < 30. Aturan ini tidak dilatih. Ia memeriksa apakah ML menambah manfaat dibandingkan sekadar meneruskan ambang yang sama dari pre-test ke post-test. Ambang 30 tetap bersifat sementara sampai disahkan pemilik instrumen/program.

Evaluasi memakai fold yang sama dengan model (Repeated Multilabel Stratified 5-Fold, 5 pengulangan). Bandingkan Macro-F1, precision, recall, Hamming loss, dan exact-match/subset accuracy; baca juga metrik per target. Jangan pilih sistem hanya dari satu metrik.


In [ ]:
# Pembanding sederhana: status post diperkirakan dari skor pre pada aspek yang sama.
# Ini aturan tetap, bukan model yang dilatih. Ambang 30 masih perlu validasi substantif.
BASELINE_THRESHOLD = 30
BASELINE_FEATURE_MAP = {
    "Y_Potensi": "Potensi_Pre",
    "Y_Eksplorasi": "Eksplorasi_Pre",
    "Y_Refleksi": "Refleksi_Pre",
    "Y_Perencanaan": "Perencanaan_Pre",
}
baseline_pred = np.column_stack([
    (X[BASELINE_FEATURE_MAP[target]] < BASELINE_THRESHOLD).astype(int).to_numpy()
    for target in TARGETS
])

baseline_fold_rows = []
for seed in range(5):
    baseline_cv = MultilabelStratifiedKFold(
        n_splits=5, shuffle=True, random_state=seed
    )
    for fold, (_, test_idx) in enumerate(baseline_cv.split(X, Y), start=1):
        fold_metrics = evaluate_predictions(
            Y.iloc[test_idx].to_numpy(), baseline_pred[test_idx]
        )
        fold_metrics.update({"seed": seed, "fold": fold})
        baseline_fold_rows.append(fold_metrics)

baseline_cv_results = pd.DataFrame(baseline_fold_rows)
metric_columns = [
    "macro_f1", "macro_precision", "macro_recall",
    "hamming_loss", "subset_accuracy",
]
model_cv_summary = cv_results[metric_columns].agg(["mean", "std"])
baseline_cv_summary = baseline_cv_results[metric_columns].agg(["mean", "std"])
baseline_oof_metrics = evaluate_predictions(Y.to_numpy(), baseline_pred)

comparison_rows = []
for name, cv_summary, oof in [
    (MODEL_NAME, model_cv_summary, oof_metrics),
    ("Aturan pre-test < 30", baseline_cv_summary, baseline_oof_metrics),
]:
    row = {"model_or_rule": name}
    for metric in metric_columns:
        row[f"{metric}_cv_mean"] = float(cv_summary.loc["mean", metric])
        row[f"{metric}_cv_std"] = float(cv_summary.loc["std", metric])
        row[f"{metric}_oof"] = float(oof[metric])
    comparison_rows.append(row)

baseline_comparison = pd.DataFrame(comparison_rows)
target_comparison_rows = []
for index, target in enumerate(TARGETS):
    y_target = Y.iloc[:, index].to_numpy()
    baseline_target = baseline_pred[:, index]
    model_target = oof_pred[:, index]
    target_comparison_rows.append({
        "target": target,
        "baseline_precision": precision_score(y_target, baseline_target, zero_division=0),
        "baseline_recall": recall_score(y_target, baseline_target, zero_division=0),
        "baseline_f1": f1_score(y_target, baseline_target, zero_division=0),
        "model_precision": precision_score(y_target, model_target, zero_division=0),
        "model_recall": recall_score(y_target, model_target, zero_division=0),
        "model_f1": f1_score(y_target, model_target, zero_division=0),
    })
target_comparison = pd.DataFrame(target_comparison_rows)

display(baseline_comparison.round(3))
display(target_comparison.round(3))
baseline_comparison.to_csv(
    REPORT_DIR / f"{safe_name}_baseline_comparison.csv", index=False
)
target_comparison.to_csv(
    REPORT_DIR / f"{safe_name}_target_comparison.csv", index=False
)
baseline_cv_results.to_csv(
    REPORT_DIR / f"{safe_name}_baseline_cv_results.csv", index=False
)


## Audit sensitivitas ambang label

Workbook source is checked at two plausible score boundaries for descriptive purposes only. Counts show how changing the target definition changes class prevalence; they do not establish which threshold is substantively correct.


In [ ]:
# Sensitivity audit of the engineered post-test labels; does not change training targets.
source_df = pd.read_excel(DATA_PATH, sheet_name="Dataset_ML", header=3)
post_score_map = {
    "Y_Potensi": "Potensi_Post",
    "Y_Eksplorasi": "Eksplorasi_Post",
    "Y_Refleksi": "Refleksi_Post",
    "Y_Perencanaan": "Perencanaan_Post",
}
threshold_sensitivity_rows = []
for target, post_column in post_score_map.items():
    scores = pd.to_numeric(source_df[post_column], errors="coerce").dropna()
    for cutoff in (20, 30):
        n_below = int((scores < cutoff).sum())
        threshold_sensitivity_rows.append({
            "target": target,
            "positive_if_post_below": cutoff,
            "n_positive": n_below,
            "n_total": int(scores.shape[0]),
            "positive_share": n_below / scores.shape[0],
        })
threshold_sensitivity = pd.DataFrame(threshold_sensitivity_rows)
display(threshold_sensitivity)
threshold_sensitivity.to_csv(
    REPORT_DIR / "posttest_threshold_sensitivity.csv", index=False
)


## Interpretasi dan batasan

- Target `Perlu` pada eksperimen ini adalah proxy berbasis skor post-test < 30; belum terbukti sama dengan keputusan intervensi pendamping.
- Model memprediksi risiko skor post-test di bawah ambang dari data pre-test. Model tidak memperkirakan efek suatu program dan tidak boleh memilih program otomatis.
- Logistic Regression adalah kandidat terbaik di antara tiga algoritma ML yang diuji, tetapi harus dibandingkan dengan aturan pre-test sederhana. Hasil baseline menunjukkan aturan tersebut dapat memiliki recall tinggi dengan lebih banyak false positive; manfaat praktis harus ditentukan bersama pendamping.
- Dataset 51 anak dan repeated cross-validation hanya memberi estimasi internal yang masih tidak pasti. Tanpa peserta baru, performa pada kohort lain belum dapat diklaim.
- Pemetaan empat target dataset ke lima komponen instrumen aplikasi dan validitas ambang perlu disahkan sebelum inferensi dimasukkan ke APK.
- Permutation importance menunjukkan perubahan skor saat fitur diacak; bukan sebab-akibat atau bukti bahwa intervensi pada fitur itu akan efektif.
- Probabilitas belum boleh disebut tingkat keyakinan yang terkalibrasi sebelum evaluasi kalibrasi yang memadai.
